# Getting Started with PySUS

*Your complete guide to accessing Brazilian public health data*

**PySUS v2.1.0 · Python 3.10+**

Notebook contribution — [AlertaDengue/PySUS](https://github.com/AlertaDengue/PySUS) · Issue #277

---

## Introduction

PySUS is a Python library that provides easy access to publicly available datasets
from Brazil's Unified Health System (SUS), published by DATASUS.
It handles file discovery, downloading, and parsing — so you can focus on data analysis
rather than dealing with legacy file formats.

This notebook presents a complete beginner-friendly workflow,
from installation to the first data exploration using real SUS data.

> **No prior knowledge of DATASUS is required.**
> All datasets are freely available at [datasus.saude.gov.br](https://datasus.saude.gov.br)
> and fetched directly from official government servers.

---
## 1. Installation

Install PySUS using `pip`.
If you are working inside a virtual environment (recommended), activate it first.

```
pip install pysus
```

> **Tip:** Using a virtual environment keeps your project dependencies isolated.
> ```
> python -m venv venv
> source venv/bin/activate      # Linux / macOS
> venv\Scripts\activate         # Windows
> ```

In [1]:
# Run this cell if PySUS is not yet installed
%pip install pysus

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 26.1.2
[notice] To update, run: C:\Users\vladi\AppData\Local\Programs\Python\Python313\python.exe -m pip install --upgrade pip


---
## 2. Checking Your Installation

After installing PySUS, verify that the package is available and check the installed version.
If the installation was successful, Python will display the version number.

In [2]:
import pysus

print(pysus.__version__)

2.2.0


---
## 3. Exploring the Package

PySUS exposes each health dataset as a simple callable function.
Use `dir(pysus)` to see everything available:

In [3]:
import pysus

print(dir(pysus))

['CACHEPATH', 'Final', '__annotations__', '__builtins__', '__cached__', '__doc__', '__file__', '__loader__', '__name__', '__package__', '__path__', '__spec__', '__version__', 'api', 'ciha', 'cnes', 'get_version', 'ibge', 'importlib_metadata', 'list_files', 'os', 'pathlib', 'pni', 'sia', 'sih', 'sim', 'sinan', 'sinasc', 'version']


The main dataset functions are:

| Function | Dataset | Description |
|----------|---------|-------------|
| `sinasc()` | SINASC | Live birth records |
| `sim()` | SIM | Mortality records |
| `sinan()` | SINAN | Notifiable diseases |
| `sih()` | SIH | Hospital admissions |
| `sia()` | SIA | Outpatient procedures |
| `cnes()` | CNES | Health facilities |
| `pni()` | PNI | Immunisation programme |
| `ibge()` | IBGE | Demographic data |

---
## 4. Understanding the Parameters

All dataset functions share the same parameter pattern.
Here is the signature for `sinasc()`:

```python
sinasc(
    state = "SP",   # two-letter Brazilian state code
    year  = 2022,   # integer or list of integers
)
```

| Parameter | Type | Description | Example |
|-----------|------|-------------|---------|
| `state` | `str` | Two-letter state abbreviation | `"SP"`, `"RJ"`, `"MG"` |
| `year` | `int` or `list[int]` | Single year or list of years | `2022` or `[2021, 2022]` |
| `group` | `str` or `None` | Sub-group code (SINAN only) | `"DENG"` (dengue) |

---
## 5. Listing Available Files

Before downloading, use `list_files()` to discover what is available
for a given dataset, state, and year.

> **Note:** Jupyter runs an async event loop internally.
> We use `nest_asyncio` to allow PySUS async calls inside the notebook.

In [4]:
# Required to run PySUS async functions inside Jupyter
import nest_asyncio
nest_asyncio.apply()

from pysus import list_files

# List SINASC files available for Rio de Janeiro, 2022
available = list_files(dataset="SINASC", state="RJ", year=2022)

print(f"Files found: {len(available)}")
available

Exception in callback Task.__step()
handle: <Handle Task.__step()>
Traceback (most recent call last):
  File "C:\Users\vladi\AppData\Local\Programs\Python\Python313\Lib\asyncio\events.py", line 89, in _run
    self._context.run(self._callback, *self._args)
    ~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
RuntimeError: cannot enter context: <_contextvars.Context object at 0x0000026B7552CC80> is already entered
Exception in callback Task.__step()
handle: <Handle Task.__step()>
Traceback (most recent call last):
  File "C:\Users\vladi\AppData\Local\Programs\Python\Python313\Lib\asyncio\events.py", line 89, in _run
    self._context.run(self._callback, *self._args)
    ~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
RuntimeError: cannot enter context: <_contextvars.Context object at 0x0000026B7552CC80> is already entered
Exception in callback Task.__step()
handle: <Handle Task.__step()>
Traceback (most recent call last):
  File "C:\Users\vladi\AppData\Local\Programs\Python\Python313\Lib\a

Files found: 1


,name,path,dataset,group,year,month,state,modify
0,public\data\ftp\sinasc\DNRJ2022.parquet,public\data\ftp\sinasc\DNRJ2022.parquet,sinasc,None,2022,None,RJ,2023-12-20 16:45:00


---
## 6. Downloading Your First Dataset

Now download SINASC birth records for Rio de Janeiro, 2022.
The function returns a **pandas DataFrame** directly —
no manual file handling required.

> **Note:** The first download may take about 30 seconds depending on your connection.
> PySUS caches files locally, so the next call for the same data will be much faster.

In [ ]:
from pysus import sinasc

# Download live birth records — Rio de Janeiro, 2022
df = sinasc(state="RJ", year=2022)

print(f"Rows   : {len(df):,}")
print(f"Columns: {df.shape[1]}")
df.head()

> **Tip:** To download multiple years at once, pass a list:
> ```python
> df = sinasc(state="SP", year=[2020, 2021, 2022])
> ```
> PySUS concatenates the results into a single DataFrame automatically.

---
## 7. Inspecting the Data

Three essential commands for exploring any new DataFrame:

In [ ]:
# Shape: number of rows and columns
print(df.shape)

In [ ]:
# Column names and data types
df.info()

In [ ]:
# Descriptive statistics for numeric columns
df.describe()

In [ ]:
# Check for missing values (top 10 columns with most nulls)
missing = df.isnull().sum().sort_values(ascending=False)
print("Columns with missing values:")
print(missing[missing > 0].head(10))

Key SINASC columns you will encounter:

| Column | Description |
|--------|-------------|
| `DTNASC` | Birth date (format: DDMMYYYY) |
| `IDADEMAE` | Mother's age in years |
| `ESCMAE` | Mother's years of schooling |
| `PARTO` | Type of delivery (1 = vaginal, 2 = caesarean) |
| `CONSULTAS` | Number of prenatal visits |
| `SEXO` | Sex of the newborn |
| `PESO` | Birth weight in grams |

---
## 8. Understanding the Local Cache

PySUS caches downloaded files locally.
The second time you request the same data, it loads from disk instead of
re-downloading — making repeated analysis much faster.

In [ ]:
import pysus

# See where PySUS stores its cached files
print(pysus.CACHEPATH)
# Typical output: ~/.pysus

> The cache directory is `~/.pysus` by default on Linux and macOS,
> and `C:\Users\<you>\.pysus` on Windows.
> You can safely delete it to free disk space;
> data will be re-downloaded on the next call.

---
## 9. Your First Visualisation

Let's plot the **monthly distribution of births** in Rio de Janeiro for 2022.

The column `DTNASC` stores the birth date in the format `DDMMYYYY`.
We extract the month from it and create a simple bar chart.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

# Parse birth month from DTNASC (format: DDMMYYYY)
df["birth_month"] = pd.to_datetime(
    df["DTNASC"], format="%d%m%Y", errors="coerce"
).dt.month

# Count births per month
monthly = df["birth_month"].value_counts().sort_index()

month_names = [
    "Jan", "Feb", "Mar", "Apr", "May", "Jun",
    "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"
]

# Plot
fig, ax = plt.subplots(figsize=(10, 5))

ax.bar(
    range(1, len(monthly) + 1),
    monthly.values,
    color="steelblue",
    edgecolor="white",
    width=0.7,
)

ax.set_xticks(range(1, 13))
ax.set_xticklabels(month_names)
ax.yaxis.set_major_formatter(
    ticker.FuncFormatter(lambda x, _: f"{int(x):,}")
)
ax.set_title("Monthly Live Births — Rio de Janeiro (2022)", fontsize=14, pad=12)
ax.set_xlabel("Month")
ax.set_ylabel("Number of births")
ax.spines[["top", "right"]].set_visible(False)

plt.tight_layout()
plt.show()

print(f"\nTotal births plotted: {monthly.sum():,}")

---
## 10. Available Data Sources

PySUS gives access to the following official DATASUS datasets:

| Dataset | Full name | Coverage |
|---------|-----------|----------|
| SINASC | Sistema de Informação sobre Nascidos Vivos | Live births, all states, 1994–present |
| SIM | Sistema de Informação sobre Mortalidade | Deaths, all states, 1979–present |
| SINAN | Sistema de Informação de Agravos de Notificação | Notifiable diseases (dengue, TB, etc.) |
| SIH | Sistema de Informações Hospitalares | Hospital admissions, 1992–present |
| SIA | Sistema de Informações Ambulatoriais | Outpatient procedures |
| CNES | Cadastro Nacional de Estabelecimentos de Saúde | Health facilities registry |
| PNI | Programa Nacional de Imunizações | Vaccination data |
| IBGE | Instituto Brasileiro de Geografia e Estatística | Population and demographic data |

---
## 11. Next Steps

You now have a working PySUS workflow. Here are some directions to explore next:

- **Analyse maternal age** — use the `IDADEMAE` column
- **Compare delivery types** — vaginal vs. caesarean using `PARTO`
- **Explore prenatal visits** — `CONSULTAS` column
- **Download mortality data** — `sim(state="SP", year=2022)`
- **Investigate notifiable diseases** — `sinan(state="RJ", year=2022, group="DENG")`
- **Compare multiple states** — loop over a list of state codes

---

**Useful resources:**

- [PySUS documentation](https://pysus.readthedocs.io)
- [PySUS GitHub repository](https://github.com/AlertaDengue/PySUS)
- [DATASUS portal](https://datasus.saude.gov.br) — official Brazilian health data portal